# LVDS CARRY4 TDC — internal self-test walkthrough

This notebook exercises the **internal** (no-cable) self-test sources of the
CARRY4 time-to-digital converter over SCPI. It does not touch the E3
connector or any external hardware, so it is safe to run against any
bitstream that includes the TDC, even before the analogue-to-time PCB
exists.

It is a teaching/bring-up notebook, not the calibration tool. For a real
code-density calibration run (hundreds of thousands to millions of events,
LUT upload, precision test, saved JSON record) use
`scpi-tests/tdc_calibrate.py` instead — this notebook keeps event counts
small on purpose so every cell finishes in a few seconds.

See `prj/v0.94/doc/tdc_registers.md` (FPGA repo) for the authoritative
register/bit-field reference.

## 1. How the internal self-test sources work

The TDC measures time in two stages:

* **Coarse**: a free-running counter clocked by `adc_clk` (125 MHz, 8 ns/tick)
  counts whole clock periods between the launch and the return event.
* **Fine**: the return event also propagates through a chain of 128 `CARRY4`
  primitives (512 taps). At the next `adc_clk` edge, the taps that the edge
  has already reached read back as `1`, the ones it has not yet reached read
  back as `0`. The position of that `1→0` transition ("raw fine index")
  encodes *where inside one 8 ns clock period* the edge arrived.

Normally the return event comes from the external LVDS loop (launch pulse
out through E3, through the analogue-to-time PCB, back in through E3). The
FPGA also implements **two internal event sources** that feed the same
carry chain without needing that cable, so the placed hardware can be
exercised on the bench first:

| Source | Register bit | What it does | What it proves |
|---|---|---|---|
| Asynchronous divided `fclk[0]` | `CONTROL` bit 10 (`INTERNAL_TEST`) | Divides an independent PS clock (`fclk[0]`) down to a slow (~61 kHz) square wave and feeds it straight into the carry chain | The placed carry cells, tap sampling flip-flops, bubble-correcting encoder, histogram, registers and SCPI/API path all work. Because `fclk[0]` and `adc_clk` are unrelated clocks, the edge's phase drifts across the whole 8 ns period over time — useful for **code-density** (histogram) calibration, but the phase at any one instant is *not* known or commanded. |
| Dedicated MMCM phase sweep | `CONTROL` bit 11 (`MMCM_SWEEP_ENABLE`) | A separate `MMCME2_ADV` (belongs only to this self-test — **not** the shared Red Pitaya PLL that clocks the ADC/DAC/serial/PWM paths) regenerates the same 125 MHz frequency at a phase you step one increment at a time (nominally ~17.86 ps/step), then releases exactly one edge into the carry chain per request | The same hardware, but at a phase **you chose**, so you can check monotonicity, repeatability and clock-boundary behaviour deterministically, one step at a time |

Both sources are purely internal hardware self-tests: they validate the
*placed silicon*, not the E3 electrical path, LVDS levels, cable, PCB delay
or the analogue-to-time conversion. Neither one calibrates absolute
picoseconds on its own — that still requires the histogram/LUT workflow
(`tdc_calibrate.py`) and, ultimately, comparison with an external calibrated
delay source.

If `MMCM_SWEEP_ENABLE` and `INTERNAL_TEST` are both set, the MMCM sweep wins.
Only one internal source (or the external E3 return) should normally be
selected at a time.

## 2. Connect

Edit `RP_HOST` below to your board's hostname or IP address
(e.g. `rp-xxxxxx.local` or `192.168.1.100`). `redpitaya_scpi.py` in this
folder is the same helper library used by the other `scpi-tests` notebooks;
the TDC does not (yet) have dedicated helper methods in that library, so
this notebook talks to it with the raw `TDC:*` SCPI commands, exactly like
`tdc_calibrate.py` does.

In [ ]:
import time

import matplotlib.pyplot as plt

import redpitaya_scpi as scpi

RP_HOST = "169.254.77.151"  # <-- set this to your board

rp = scpi.scpi(RP_HOST, timeout=10.0)
print('connected to', RP_HOST)

In [ ]:
# Small helpers mirroring the ones used in scpi-tests/tdc_calibrate.py.
# TDC:*? queries return a comma-separated list of integers (SCPI_ResultMnemonic).

def query_csv(command):
    """Send a query and parse a comma-separated integer response."""
    text = rp.txrx_txt(command).strip().strip('{}')
    if not text:
        return []
    return [int(item.strip(), 0) for item in text.split(',')]


def send(command):
    """Send a command with no response expected."""
    rp.tx_txt(command)


# Per-result status bits, from rp.h / tdc_registers.md.
RESULT_VALID                = 1 << 0
RESULT_NO_TRANSITION         = 1 << 1
RESULT_MULTIPLE_TRANSITIONS = 1 << 2
RESULT_ALL_ZERO             = 1 << 3
RESULT_ALL_ONE              = 1 << 4
RESULT_BUBBLE_CORRECTED     = 1 << 5
RESULT_TIMEOUT              = 1 << 6
RESULT_FIFO_OVERFLOW        = 1 << 7

RESULT_FLAG_NAMES = {
    RESULT_VALID: 'VALID',
    RESULT_NO_TRANSITION: 'NO_TRANSITION',
    RESULT_MULTIPLE_TRANSITIONS: 'MULTIPLE_TRANSITIONS',
    RESULT_ALL_ZERO: 'ALL_ZERO',
    RESULT_ALL_ONE: 'ALL_ONE',
    RESULT_BUBBLE_CORRECTED: 'BUBBLE_CORRECTED',
    RESULT_TIMEOUT: 'TIMEOUT',
    RESULT_FIFO_OVERFLOW: 'FIFO_OVERFLOW',
}


def describe_status(status):
    """Turn a per-result status byte into a readable flag list."""
    names = [name for bit, name in RESULT_FLAG_NAMES.items() if status & bit]
    return ','.join(names) if names else 'none'


def measure_once():
    """Trigger one single-shot measurement and return the parsed TDC:MEASure? row:
    [sequence, status, coarse_cycles, fine_raw, fine_ps, total_ps, analogue_code]
    Assumes TDC:MODE SINGLE and TDC:ENABle ON have already been set."""
    result = query_csv('TDC:MEASure?')
    assert len(result) == 7, 'unexpected TDC:MEASure? response: {}'.format(result)
    return result

## 3. Reset and basic identification

`TDC:RESet` clears the controller/counters/FIFO. It deliberately preserves
which internal source is selected (bits 10/11), so you can reset between
measurements without losing your `TDC:TEST:INTernal` / `TDC:MMCM:ENABle`
choice.

`TDC:TAPs?` reading back `512` is a quick end-to-end proof that the register
map/signature check and system-bus routing are working before doing anything
more elaborate.

In [ ]:
send('TDC:RESet')

taps = int(rp.txrx_txt('TDC:TAPs?'))
print('carry-chain taps:', taps)
assert taps == 512, 'unexpected tap count -- is the bitstream/TDC actually loaded?'

## 4. Asynchronous `fclk[0]` internal source

Command sequence for one single-shot self-test result:

```text
TDC:RESet
TDC:TEST:INTernal ON     # select the divided-fclk[0] source (CONTROL bit 10)
TDC:MODE SINGLE
TDC:ENABle ON            # arm; the controller now waits for the next internal edge
TDC:MEASure?              # triggers + blocks (via FIFO polling) until a result exists
```

`TDC:MEASure?` (implemented server-side in `scpi-commands.cpp`) issues the
trigger and polls the FIFO for you, so a single query is enough. Because this
source waits for the next *external-to-adc_clk* rising edge rather than
driving anything, `total_time_ps` in the reply is dominated by an arbitrary
wait and is **not** meaningful here — look at `status`, `fine_raw` and
`fine_ps` instead.

In [ ]:
send('TDC:RESet')
send('TDC:TEST:INTernal ON')
print('internal (async fclk[0]) source selected:', rp.txrx_txt('TDC:TEST:INTernal?'))

send('TDC:MODE SINGLE')
send('TDC:ENABle ON')

sequence, status, coarse, fine_raw, fine_ps, total_ps, analogue_code = measure_once()
print('sequence       :', sequence)
print('status flags   :', describe_status(status))
print('coarse cycles  :', coarse)
print('raw fine index :', fine_raw, '/', taps)
print('fine time (ps) :', fine_ps, '(diagnostic average-bin value; no LUT uploaded yet)')

send('TDC:ENABle OFF')

In [ ]:
# Repeat a handful of single-shot measurements and watch the raw fine index move.
# Because fclk[0] and adc_clk are independent, unrelated clocks, the sampled phase
# (and therefore the raw fine index) should walk around rather than staying fixed.

send('TDC:ENABle ON')
raw_indices = []
for _ in range(10):
    result = measure_once()
    raw_indices.append(result[3])
send('TDC:ENABle OFF')

print('raw fine indices over 10 samples:', raw_indices)
print('(a fixed synchronous loopback would instead repeat the same few bins --')
print(' see HANDOVER section 16.1 on why phase coverage matters for calibration)')

## 5. Code-density histogram with the async source

`TDC:CALibration:STARt <events>` puts the TDC into calibration mode: it
accepts asynchronous events from whichever internal/external source is
currently selected and increments a per-tap histogram bin for every valid
transition, without pushing each one into the (small, 256-entry) measurement
FIFO. If the histogram populates many bins roughly evenly, that demonstrates
the source has enough phase coverage to be a genuine calibration source (see
HANDOVER section 16).

This cell only collects a **small** number of events for a quick,
notebook-friendly demonstration of the mechanism. For a real calibration,
with a properly sized LUT upload and precision test, use
`scpi-tests/tdc_calibrate.py --internal --events 1000000` instead.

In [ ]:
CALIBRATION_COMPLETE = 1 << 17

demo_target_events = 5000  # keep this small just to see the shape quickly

send('TDC:TEST:INTernal ON')  # make sure the async source is still selected
send('TDC:CALibration:STARt {}'.format(demo_target_events))

deadline = time.monotonic() + 60.0
status, events = 0, 0
while time.monotonic() < deadline:
    status, events = query_csv('TDC:CALibration:STATus?')
    if status & CALIBRATION_COMPLETE:
        break
    time.sleep(0.1)
send('TDC:CALibration:STOP')
print('collected', events, 'valid calibration events')

histogram = query_csv('TDC:CALibration:HISTogram? 0,{}'.format(taps))
assert len(histogram) == taps

populated_bins = sum(1 for count in histogram if count > 0)
print('populated bins: {} / {}'.format(populated_bins, taps))

plt.figure(figsize=(9, 3))
plt.bar(range(taps), histogram, width=1.0)
plt.xlabel('raw tap index')
plt.ylabel('event count')
plt.title('Code-density histogram (async fclk[0] source, {} events)'.format(events))
plt.tight_layout()
plt.show()

## 6. Dedicated MMCM phase-sweep source

This source lets you *command* a phase rather than waiting for one to drift
past. Sequence:

```text
TDC:RESet
TDC:MMCM:ENABle ON        # select the MMCM source (CONTROL bit 11, takes
                          # priority over bit 10 if both are set)
TDC:MMCM:STATus?           # poll bit 0 (LOCKED) until it is 1
TDC:MMCM:STEP INCREASE     # request ONE dynamic phase-shift step (~17.86 ps
                          # nominal); blocks server-side until PSDONE or a
                          # timeout -- see rp_TdcMmcmPhaseStep()
TDC:MODE SINGLE
TDC:ENABle ON              # arm; the MMCM stimulus fires exactly one edge
TDC:MEASure?                # read the result at this commanded phase
TDC:ENABle OFF
# ... repeat STEP/MODE/ENABle/MEASure for each phase position ...
TDC:MMCM:PHASe:COUNt?      # signed accumulated step count, for bookkeeping
```

Unlike the async source, `TDC:MMCM:STEP` moves the edge by a *known* nominal
amount each time, so plotting raw fine index against step number should show
a predominantly monotonic staircase (allowing for jitter near bin
boundaries) -- useful for checking monotonicity and clock-boundary behaviour
without any external hardware. `TDC:MMCM:COARse:DELay <cycles>` additionally
adds whole 8 ns `clk_i` cycles before the edge fires, if you want to sweep
across several clock periods.

As with the async source, this does not exercise the E3 electrical path and
the phase-step size is nominal (real skew/jitter apply) -- it characterizes
response to a *commanded* phase, it does not certify absolute picoseconds.

In [ ]:
MMCM_STATUS_LOCKED     = 1 << 0
MMCM_STATUS_PS_BUSY    = 1 << 1
MMCM_STATUS_PS_TIMEOUT = 1 << 2

send('TDC:RESet')
send('TDC:MMCM:ENABle ON')

# The MMCM needs a moment to lock after being selected/reset; poll rather
# than assuming a fixed delay.
deadline = time.monotonic() + 5.0
locked = False
while time.monotonic() < deadline:
    mmcm_status = int(rp.txrx_txt('TDC:MMCM:STATus?'))
    locked = bool(mmcm_status & MMCM_STATUS_LOCKED)
    if locked:
        break
    time.sleep(0.05)
print('MMCM locked:', locked)
assert locked, 'dedicated MMCM did not lock -- check the bitstream/clock topology'

In [ ]:
# Sweep a modest number of phase steps, taking one single-shot measurement at
# each commanded position. Each TDC:MMCM:STEP blocks server-side until the
# FPGA acknowledges PSDONE (or times out), so no extra polling is needed here.

num_steps = 40
phase_counts = []
raw_indices = []

send('TDC:MODE SINGLE')
for step in range(num_steps):
    send('TDC:MMCM:STEP INCREASE')

    mmcm_status = int(rp.txrx_txt('TDC:MMCM:STATus?'))
    if mmcm_status & MMCM_STATUS_PS_TIMEOUT:
        print('warning: phase step', step, 'timed out; clearing and continuing')
        send('TDC:MMCM:TIMeout:CLEar')

    send('TDC:ENABle ON')
    result = measure_once()
    send('TDC:ENABle OFF')

    phase_count = int(rp.txrx_txt('TDC:MMCM:PHASe:COUNt?'))
    phase_counts.append(phase_count)
    raw_indices.append(result[3] if result[1] & RESULT_VALID else None)

print('commanded phase steps:', phase_counts)
print('raw fine indices     :', raw_indices)

plt.figure(figsize=(9, 3))
valid_x = [p for p, r in zip(phase_counts, raw_indices) if r is not None]
valid_y = [r for r in raw_indices if r is not None]
plt.plot(valid_x, valid_y, marker='o')
plt.xlabel('commanded phase-step count (nominal ~17.86 ps/step)')
plt.ylabel('raw fine tap index')
plt.title('MMCM phase sweep: raw code vs. commanded phase')
plt.tight_layout()
plt.show()

## 7. Clean up

Always disable the internal test sources before switching back to a real
external-E3 measurement, and before closing the connection -- otherwise the
TDC stays armed against a source you no longer intend to use.

In [ ]:
send('TDC:ENABle OFF')
send('TDC:MMCM:ENABle OFF')
send('TDC:TEST:INTernal OFF')
send('TDC:RESet')

rp.close()
print('disconnected')